In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_0)

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
# Config
MODEL_NAME = "roberta-base"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
SEED       = 42
SAVE_DIR   = "./model_output/emotion_detection"

In [4]:
# Dataset paths
BASE_PATH = "/kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/emotion-detection"

PATH_TRAIN = f"{BASE_PATH}/train.csv"
PATH_VAL   = f"{BASE_PATH}/val.csv"

In [5]:
# Emotion label mapping (5 unified classes)
EMOTION_CATEGORIES = ['angry', 'anxious', 'content', 'joyful', 'sad']
label2id = {l: i for i, l in enumerate(EMOTION_CATEGORIES)}
id2label = {i: l for l, i in label2id.items()}

print("Label mapping:", label2id)

Label mapping: {'angry': 0, 'anxious': 1, 'content': 2, 'joyful': 3, 'sad': 4}


In [6]:
# Load pre-split train and val CSVs
train_df = pd.read_csv(PATH_TRAIN)
val_df   = pd.read_csv(PATH_VAL)

# Safety filter — keep only the 5 known classes
train_df = train_df[train_df["label"].isin(EMOTION_CATEGORIES)].reset_index(drop=True)
val_df   = val_df[val_df["label"].isin(EMOTION_CATEGORIES)].reset_index(drop=True)

# Encode string labels -> integers
train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"]   = val_df["label"].map(label2id)

print(f"Train : {len(train_df)} rows")
print(train_df["label"].value_counts())
print(f"\nVal   : {len(val_df)} rows")
print(val_df["label"].value_counts())

Train : 19059 rows
label
sad        5240
angry      4454
joyful     3637
anxious    2938
content    2790
Name: count, dtype: int64

Val   : 2380 rows
label
sad        655
angry      556
joyful     454
anxious    367
content    348
Name: count, dtype: int64


In [7]:
# Tokenizer & Dataset class
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts.tolist(),
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels.tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }

train_dataset = EmotionDataset(train_df["text"], train_df["label_id"])
val_dataset   = EmotionDataset(val_df["text"],   val_df["label_id"])

print(f"Train dataset : {len(train_dataset)} samples")
print(f"Val dataset   : {len(val_dataset)} samples")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train dataset : 19059 samples
Val dataset   : 2380 samples


In [8]:
# Model
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTION_CATEGORIES),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Training arguments
# Val is used only to track loss and pick the best checkpoint.
# All metrics (accuracy, F1, confusion matrix) are computed in the evaluation notebook.
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_dir="./logs",
    logging_steps=50,
    seed=SEED,
    fp16=torch.cuda.is_available()
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# Trainer & training run
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.937682,1.907252
2,1.656292,1.770591
3,1.384891,1.778660
4,1.126796,1.907596
5,0.927259,1.955696


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2980, training_loss=1.4603391429721908, metrics={'train_runtime': 1400.7814, 'train_samples_per_second': 68.03, 'train_steps_per_second': 2.127, 'total_flos': 6268460846526720.0, 'train_loss': 1.4603391429721908, 'epoch': 5.0})

In [11]:
# Save model & tokenizer locally
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./model_output/emotion_detection


In [12]:
# Push to HuggingFace Hub
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_0)

model.push_to_hub("melisaolivia18/emotion-detection-roberta")
tokenizer.push_to_hub("melisaolivia18/emotion-detection-roberta")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/melisaolivia18/emotion-detection-roberta/commit/2220d72bef6e42b90a4f289172e67d46fcd0651f', commit_message='Upload tokenizer', commit_description='', oid='2220d72bef6e42b90a4f289172e67d46fcd0651f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/melisaolivia18/emotion-detection-roberta', endpoint='https://huggingface.co', repo_type='model', repo_id='melisaolivia18/emotion-detection-roberta'), pr_revision=None, pr_num=None)